# Vaidya — Phase 5: Quantization + Benchmark

**Steps:**
1. Install packages
2. Verify GPU
3. Set working directory + AWS creds
4. GPTQ 4-bit quantization
5. AWQ 4-bit quantization
6. GGUF conversion (Q4_K_M, Q5_K_M, Q8_0)
7. Benchmark all formats → table + S3

**GPU:** L4 (24GB) works for GPTQ/AWQ. GGUF conversion can run on CPU (just slower).  
**Time:** ~30 min total (GPTQ 10 min, AWQ 10 min, GGUF 5 min, benchmark 10 min)

## Step 1 — Install packages

In [ ]:
# Core packages
!pip install -q \
    transformers==4.47.0 \
    peft==0.14.0 \
    accelerate==0.34.2 \
    datasets==3.2.0 \
    boto3 \
    sentencepiece \
    pandas \
    "fsspec[http]==2024.9.0" \
    "packaging==24.2"

# GPTQ
!pip install -q auto-gptq --extra-index-url https://huggingface.github.io/autogptq-index/whl/cu121/

# AWQ
!pip install -q autoawq

# GGUF inference (for benchmark)
!pip install -q llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu121

print('✅ Install done')
print('⚠️  RESTART THE KERNEL, then run from Step 2.')

## Step 2 — Verify GPU

In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU — check your Lightning.ai machine'
print(f'GPU  : {torch.cuda.get_device_name(0)}')
print(f'VRAM : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
print(f'CC   : {torch.cuda.get_device_capability()}')

## Step 3 — Working directory + AWS credentials

In [ ]:
import os
work_dir = '/teamspace/studios/this_studio/vaidya/layer4_optimize'
os.chdir(work_dir)
print(f'Working dir: {os.getcwd()}')

In [ ]:
import os, boto3

# ── Fill in your credentials ────────────────────────────────
os.environ['AWS_ACCESS_KEY_ID']     = 'YOUR_ACCESS_KEY_ID'
os.environ['AWS_SECRET_ACCESS_KEY'] = 'YOUR_SECRET_ACCESS_KEY'
os.environ['AWS_DEFAULT_REGION']    = 'ap-south-1'
# ────────────────────────────────────────────────────────────

s3 = boto3.client('s3')
buckets = [b['Name'] for b in s3.list_buckets()['Buckets']]
assert 'vaidya-artifacts' in buckets
print('✅ S3 connected')

## Step 4 — GPTQ 4-bit quantization

Calibrates on 128 val samples, saves to `./gptq-4bit/` → S3.  
~10 min on L4.

In [ ]:
!python quantize_gptq.py

## Step 5 — AWQ 4-bit quantization

Calibrates on 128 val samples, saves to `./awq-4bit/` → S3.  
~10 min on L4.

In [ ]:
!python quantize_awq.py

## Step 6 — GGUF conversion (llama.cpp)

Produces Q4_K_M (default) + Q5_K_M + Q8_0, saves to `./gguf-models/` → S3.  
Builds llama.cpp from source on first run (~5 min).

In [ ]:
!bash convert_gguf.sh Q4_K_M

## Step 7 — Benchmark all formats

Runs 50 generations per format, prints latency table, saves `benchmark_results.json` → S3.

In [ ]:
!python benchmark.py --n-runs 50 --formats merged gptq awq gguf

In [ ]:
# Display benchmark results as a DataFrame
import json, pandas as pd
from pathlib import Path

results = json.loads(Path('benchmark_results.json').read_text())
rows = [
    {'format': k, **{m: v for m, v in v.items()}}
    for k, v in results.items() if v
]
df = pd.DataFrame(rows).set_index('format')
df['speedup_vs_bf16'] = (df.loc['merged_bf16', 'p50_ms'] / df['p50_ms']).round(2)
df['size_reduction'] = (df.loc['merged_bf16', 'size_mb'] / df['size_mb']).round(2)
print(df.to_string())

## Gate check — before Phase 6

- ✅ GPTQ model at `s3://vaidya-artifacts/models/gptq-4bit/`
- ✅ AWQ model at `s3://vaidya-artifacts/models/awq-4bit/`
- ✅ GGUF files at `s3://vaidya-artifacts/models/gguf/`
- ✅ Benchmark results at `s3://vaidya-artifacts/models/benchmark_results.json`
- ✅ Size reduction vs BF16 merged: ~4–5×
- ✅ AWQ/GGUF speedup vs BF16: ~2–3×

**Next:** Phase 6 — Docker Compose serving stack (FastAPI + vLLM/Ollama + Prometheus + Grafana)